In [ ]:
# Goal: figure out which selector(s) actually hold the statement body.
import time
import requests
from bs4 import BeautifulSoup

# Sample URLs across eras. Note: CSV starts at 2011, so true "2003-2005" era
# is not available -- using 2011 as the oldest sample.
SAMPLES = [
    ("recent", "2025-01-29", "https://www.federalreserve.gov/newsevents/pressreleases/monetary20250129a.htm"),
    ("mid",    "2013-06-19", "https://www.federalreserve.gov/newsevents/pressreleases/monetary20130619a.htm"),
    ("old",    "2011-01-26", "https://www.federalreserve.gov/newsevents/pressreleases/monetary20110126a.htm"),
]

CANDIDATE_SEL = "div.col-xs-12.col-sm-8.col-md-8"
HEADERS = {"User-Agent": "Mozilla/5.0 (research; fomc-sentiment-spy)"}

In [2]:
# Score every container-ish tag by total text length of contained <p>.
# Returns (tag, class, char_count, sample_text) for the winner.
def best_text_container(soup):
    best = (None, None, 0, "")
    for el in soup.find_all(["div", "td", "article", "section", "main"]):
        ps = el.find_all("p", recursive=True)
        if not ps:
            continue
        txt = "\n".join(p.get_text(" ", strip=True) for p in ps)
        if len(txt) > best[2]:
            cls = ".".join(el.get("class", []) or [])
            best = (el.name, cls, len(txt), txt[:200])
    return best


def diagnose(label, date, url):
    print(f"\n=== {label.upper()} | {date} ===")
    print(f"URL: {url}")
    r = requests.get(url, headers=HEADERS, timeout=15)
    print(f"HTTP status: {r.status_code}")
    if r.status_code != 200:
        return
    soup = BeautifulSoup(r.text, "html.parser")
    cand = soup.select(CANDIDATE_SEL)
    print(f"Candidate selector matches: {len(cand)}")
    all_p = soup.find_all("p")
    print(f"Total <p> tags: {len(all_p)}")
    tag, cls, n, head = best_text_container(soup)
    print(f"Best container: <{tag} class='{cls}'> chars={n}")
    print(f"First 200 chars: {head!r}")

In [3]:
# Run probe with polite 1s delay.
for label, date, url in SAMPLES:
    diagnose(label, date, url)
    time.sleep(1.0)


=== RECENT | 2025-01-29 ===
URL: https://www.federalreserve.gov/newsevents/pressreleases/monetary20250129a.htm


HTTP status: 200
Candidate selector matches: 2
Total <p> tags: 55
Best container: <div class='container.container__main'> chars=2169
First 200 chars: 'January 29, 2025\nFor release at 2:00 p.m. EST Share\nRecent indicators suggest that economic activity has continued to expand at a solid pace. The unemployment rate has stabilized at a low level in rec'



=== MID | 2013-06-19 ===
URL: https://www.federalreserve.gov/newsevents/pressreleases/monetary20130619a.htm


HTTP status: 200
Candidate selector matches: 2
Total <p> tags: 55
Best container: <div class='container.container__main'> chars=4744
First 200 chars: 'June 19, 2013\nFor immediate release Share\nInformation received since the Federal Open Market Committee met in May suggests that economic activity has been expanding at a moderate pace. Labor market co'



=== OLD | 2011-01-26 ===
URL: https://www.federalreserve.gov/newsevents/pressreleases/monetary20110126a.htm


HTTP status: 200
Candidate selector matches: 2
Total <p> tags: 54
Best container: <div class='container.container__main'> chars=2942
First 200 chars: 'January 26, 2011\nFor immediate release Share\nInformation received since the Federal Open Market Committee met in December confirms that the economic recovery is continuing, though at a rate that has b'


In [ ]:
# for each sample so we can pin down head/tail cleaning markers verbatim.

MAIN_SEL = "div.container__main"


def extract_longest(soup):
    matches = soup.select(MAIN_SEL)
    best_el, best_txt = None, ""
    for el in matches:
        ps = el.find_all("p", recursive=True)
        txt = "\n".join(p.get_text(" ", strip=True) for p in ps)
        if len(txt) > len(best_txt):
            best_el, best_txt = el, txt
    return len(matches), best_el, best_txt


for label, date, url in SAMPLES:
    print("\n" + "=" * 78)
    print(f"{label.upper()} | {date} | {url}")
    print("=" * 78)
    r = requests.get(url, headers=HEADERS, timeout=15)
    print(f"HTTP {r.status_code}")
    if r.status_code != 200:
        
        continue
    soup = BeautifulSoup(r.text, "html.parser")
    n, el, body = extract_longest(soup)
    cls = ".".join(el.get("class", []) or []) if el is not None else None
    print(f"{MAIN_SEL} matches={n} | chosen <{el.name} class='{cls}'> | chars={len(body)}")
    print("--- FULL TEXT (verbatim) ---")
    print(body)
    print("--- END ---")
    time.sleep(1.0)


RECENT | 2025-01-29 | https://www.federalreserve.gov/newsevents/pressreleases/monetary20250129a.htm


HTTP 200
div.container__main matches=1 | chosen <div class='container.container__main'> | chars=2169
--- FULL TEXT (verbatim) ---
January 29, 2025
For release at 2:00 p.m. EST Share
Recent indicators suggest that economic activity has continued to expand at a solid pace. The unemployment rate has stabilized at a low level in recent months, and labor market conditions remain solid. Inflation remains somewhat elevated.
The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the longer run. The Committee judges that the risks to achieving its employment and inflation goals are roughly in balance. The economic outlook is uncertain, and the Committee is attentive to the risks to both sides of its dual mandate.
In support of its goals, the Committee decided to maintain the target range for the federal funds rate at 4-1/4 to 4-1/2 percent. In considering the extent and timing of additional adjustments to the target range for the federal funds rate, the Co


MID | 2013-06-19 | https://www.federalreserve.gov/newsevents/pressreleases/monetary20130619a.htm
HTTP 200
div.container__main matches=1 | chosen <div class='container.container__main'> | chars=4744
--- FULL TEXT (verbatim) ---
June 19, 2013
For immediate release Share
Information received since the Federal Open Market Committee met in May suggests that economic activity has been expanding at a moderate pace. Labor market conditions have shown further improvement in recent months, on balance, but the unemployment rate remains elevated. Household spending and business fixed investment advanced, and the housing sector has strengthened further, but fiscal policy is restraining economic growth. Partly reflecting transitory influences, inflation has been running below the Committee's longer-run objective, but longer-term inflation expectations have remained stable.
Consistent with its statutory mandate, the Committee seeks to foster maximum employment and price stability. The Committee expe


OLD | 2011-01-26 | https://www.federalreserve.gov/newsevents/pressreleases/monetary20110126a.htm
HTTP 200
div.container__main matches=1 | chosen <div class='container.container__main'> | chars=2942
--- FULL TEXT (verbatim) ---
January 26, 2011
For immediate release Share
Information received since the Federal Open Market Committee met in December confirms that the economic recovery is continuing, though at a rate that has been insufficient to bring about a significant improvement in labor market conditions. Growth in household spending picked up late last year, but remains constrained by high unemployment, modest income growth, lower housing wealth, and tight credit. Business spending on equipment and software is rising, while investment in nonresidential structures is still weak. Employers remain reluctant to add to payrolls. The housing sector continues to be depressed. Although commodity prices have risen, longer-term inflation expectations have remained stable, and measures of und

In [ ]:
#   - container = longest-text element of div.container__main
#   - head cut  = drop everything up to and including the first line ending with 'Share'
#   - tail cut  = truncate at regex r'Voting for (?:the )?(?:FOMC )?monetary policy action were'
#   - encoding  = force UTF-8 + NFKC normalize (fixes mojibake e.g. 'mortgageâbacked')

import re
import unicodedata
import pandas as pd

URLS_CSV = "/workspaces/fomc-sentiment-spy/data/processed/fomc_urls.csv"
OUT_CSV  = "/workspaces/fomc-sentiment-spy/data/processed/fomc_statements.csv"
SEL = "div.container__main"
SLEEP_SEC = 1.0

# Drop entire prefix up to and including the first line ending with 'Share'.
HEAD_RE = re.compile(r"\A.*?\bShare\s*\n", re.DOTALL)
# Marker varies across eras (2025 drops 'FOMC' + colon).
TAIL_RE = re.compile(r"Voting for (?:the )?(?:FOMC )?monetary policy action were", re.IGNORECASE)

In [6]:
def fetch(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    # Fed pages are UTF-8 but Content-Type sometimes lacks charset, causing
    # requests to fall back to ISO-8859-1. Force UTF-8 to avoid mojibake.
    r.encoding = "utf-8"
    return r


def extract_body(soup):
    matches = soup.select(SEL)
    if not matches:
        return None
    best, best_n = None, 0
    for el in matches:
        txt = "\n".join(p.get_text(" ", strip=True) for p in el.find_all("p"))
        if len(txt) > best_n:
            best, best_n = txt, len(txt)
    return best if best_n > 0 else None


def clean(raw):
    if not raw:
        return ""
    txt = unicodedata.normalize("NFKC", raw)
    m = HEAD_RE.match(txt)
    if m:
        txt = txt[m.end():]
    m = TAIL_RE.search(txt)
    if m:
        txt = txt[: m.start()]
    txt = re.sub(r"\n{2,}", "\n", txt).strip()
    return txt

In [7]:
# Main loop. Failures recorded but loop continues.
df_urls = pd.read_csv(URLS_CSV)
print(f"loaded {len(df_urls)} URLs; columns={list(df_urls.columns)}")

rows, failed = [], []
for i, r in df_urls.iterrows():
    date, url = r["date"], r["url"]
    raw, cleaned, err = "", "", ""
    try:
        resp = fetch(url)
        if resp.status_code != 200:
            err = f"HTTP {resp.status_code}"
        else:
            soup = BeautifulSoup(resp.text, "html.parser")
            raw = extract_body(soup) or ""
            if not raw:
                err = "no container/text"
            else:
                cleaned = clean(raw)
    except Exception as e:
        err = repr(e)
    if err:
        failed.append((date, url, err))
    rows.append({"date": date, "url": url, "raw_text": raw, "clean_text": cleaned})
    time.sleep(SLEEP_SEC)
    if (i + 1) % 25 == 0:
        print(f"  ...{i + 1}/{len(df_urls)} done")

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print(f"saved {len(out)} rows -> {OUT_CSV}")
print(f"failures: {len(failed)}")
for d, u, e in failed:
    print("  FAIL", d, u, "|", e)

loaded 129 URLs; columns=['date', 'url', 'link_text']


  ...25/129 done


  ...50/129 done


  ...75/129 done


  ...100/129 done


  ...125/129 done


saved 129 rows -> /workspaces/fomc-sentiment-spy/data/processed/fomc_statements.csv
failures: 0


In [8]:
# Audit: word-count distribution, flag outliers, show extremes.
wc = out["clean_text"].str.split().str.len().fillna(0).astype(int)
success = (wc > 0).sum()
print(f"total={len(out)}  success={success}  failed={len(out) - success}")
ok = wc[wc > 0]
if len(ok):
    print(f"word count (success only): min={ok.min()}  median={int(ok.median())}  max={ok.max()}")

flagged = out[(wc < 100) | (wc > 1500)].copy()
flagged["words"] = wc[flagged.index]
print(f"\nFLAGGED (<100 or >1500 words): {len(flagged)}")
for _, r in flagged.iterrows():
    print(f"  {r['date']}  words={r['words']}  {r['url']}")

order = wc.sort_values()
print("\nshortest 3:")
for idx in order.index[:3]:
    print(f"  {out.loc[idx, 'date']}  words={wc[idx]}  | {out.loc[idx, 'clean_text'][:120]!r}")
print("\nlongest 3:")
for idx in order.index[-3:]:
    print(f"  {out.loc[idx, 'date']}  words={wc[idx]}  | {out.loc[idx, 'clean_text'][:120]!r}")

total=129  success=129  failed=0
word count (success only): min=86  median=412  max=798

FLAGGED (<100 or >1500 words): 1
  2020-03-03  words=86  https://www.federalreserve.gov/newsevents/pressreleases/monetary20200303a.htm

shortest 3:
  2020-03-03  words=86  | 'The fundamentals of the U.S. economy remain strong. However, the coronavirus poses evolving risks to economic activity. '
  2025-08-22  words=196  | 'The Federal Open Market Committee (FOMC) on Friday announced the unanimous approval of updates to its Statement on Longe'
  2026-01-28  words=222  | 'Available indicators suggest that economic activity has been expanding at a solid pace. Job gains have remained low, and'

longest 3:
  2013-12-18  words=779  | 'Information received since the Federal Open Market Committee met in October indicates that economic activity is expandin'
  2014-01-29  words=790  | 'Information received since the Federal Open Market Committee met in December indicates that growth in economic activity '
  